In [0]:
dbutils.widgets.removeAll()

In [0]:
from datetime import datetime, timezone

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

dbutils.widgets.text(
    "ingestion_timestamp",
    datetime.now(timezone.utc).isoformat(),
    "Ingestion Timestamp"
)

environment = dbutils.widgets.get("environment").lower()

ingestion_timestamp = (
    dbutils.widgets.get("ingestion_timestamp").strip()
)

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "storage_account": "stcentralusjrdev",
        "catalog": "salescsv_dev"
    },
    "prod": {
        "storage_account": "stcentralusjrprod",
        "catalog": "salescsv_prod"
    }
}

env = config[environment]

storage_account = env["storage_account"]
catalog = env["catalog"]

silver_table = (
    f"{catalog}.silver.inventory_movements"
)

product_table = (
    f"{catalog}.gold.inventory_by_product"
)

warehouse_table = (
    f"{catalog}.gold.inventory_by_warehouse"
)

low_stock_table = (
    f"{catalog}.gold.low_stock_products"
)

product_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salescsv/gold/inventory_by_product/"
)

warehouse_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salescsv/gold/inventory_by_warehouse/"
)

low_stock_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salescsv/gold/low_stock_products/"
)

print("=" * 60)
print("SALES CSV - GOLD ANALYTICS")
print("=" * 60)
print(f"Environment        : {environment}")
print(f"Source             : {silver_table}")
print(f"Product target     : {product_table}")
print(f"Warehouse target   : {warehouse_table}")
print(f"Low stock target   : {low_stock_table}")
print("=" * 60)

In [0]:


silver_df = spark.table(silver_table)

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    sum,
    countDistinct,
    avg,
    min,
    max,
    round,
    lit
)


product_inventory_df = (
    silver_df

    .groupBy(
        "product_id",
        "product_name",
        "category",
        "supplier",
        "unit_cost",
        "reorder_level"
    )

    .agg(
        sum(
            when(
                col("transaction_type") == "RECEIPT",
                col("quantity")
            ).otherwise(0)
        ).alias("total_received"),

        sum(
            when(
                col("transaction_type") == "SALE",
                col("quantity")
            ).otherwise(0)
        ).alias("total_sold"),

        sum(
            when(
                col("transaction_type") == "RETURN",
                col("quantity")
            ).otherwise(0)
        ).alias("total_returned"),

        sum(
            when(
                col("transaction_type") == "ADJUSTMENT",
                col("quantity")
            ).otherwise(0)
        ).alias("total_adjustment"),

        sum("inventory_change")
            .alias("current_stock"),

        countDistinct("warehouse")
            .alias("warehouse_count")
    )

    .withColumn(
        "inventory_value",
        round(
            col("current_stock")
            * col("unit_cost"),
            2
        )
    )

    .withColumn(
        "stock_status",
        when(
            col("current_stock") <= 0,
            "OUT_OF_STOCK"
        )
        .when(
            col("current_stock") <= col("reorder_level"),
            "LOW_STOCK"
        )
        .otherwise(
            "IN_STOCK"
        )
    )

    .withColumn(
        "gold_processing_timestamp",
        lit(ingestion_timestamp).cast("timestamp")
    )
)

In [0]:
warehouse_inventory_df = (
    silver_df

    .groupBy(
        "warehouse"
    )

    .agg(
        countDistinct("product_id")
            .alias("total_products"),

        countDistinct("transaction_id")
            .alias("total_transactions"),

        sum("inventory_change")
            .alias("total_units"),

        round(
            sum("inventory_value_change"),
            2
        ).alias("inventory_value"),

        round(
            avg("inventory_change"),
            2
        ).alias("average_inventory_change"),

        min("inventory_change")
            .alias("min_inventory_change"),

        max("inventory_change")
            .alias("max_inventory_change")
    )

    .withColumn(
        "gold_processing_timestamp",
        lit(ingestion_timestamp).cast("timestamp")
    )
)

In [0]:
low_stock_df = (
    product_inventory_df

    .filter(
        col("current_stock")
        <= col("reorder_level")
    )

    .select(
        "product_id",
        "product_name",
        "category",
        "supplier",
        "unit_cost",
        "reorder_level",
        "current_stock",
        "inventory_value",
        "stock_status",
        "gold_processing_timestamp"
    )
)

In [0]:
from delta.tables import DeltaTable


def merge_gold_snapshot(
    source_df,
    target_table,
    target_path,
    merge_condition
):
    """
    Creates an external Gold Delta table during the initial load.

    Subsequent executions synchronize the Gold table with
    the latest analytical snapshot using Delta MERGE.
    """

    if not spark.catalog.tableExists(target_table):

        print(
            f"Initial load. Creating Gold table: "
            f"{target_table}"
        )

        (
            source_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .option("path", target_path)
                .saveAsTable(target_table)
        )

    else:

        print(
            f"Synchronizing Gold table: "
            f"{target_table}"
        )

        target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target.alias("target")

                .merge(
                    source_df.alias("source"),
                    merge_condition
                )

                .withSchemaEvolution()

                .whenMatchedUpdateAll()

                .whenNotMatchedInsertAll()

                .whenNotMatchedBySourceDelete()

                .execute()
        )

    print(
        f"Gold synchronization completed: "
        f"{target_table}"
    )

In [0]:
merge_gold_snapshot(
    source_df=product_inventory_df,
    target_table=product_table,
    target_path=product_path,
    merge_condition=(
        "target.product_id = source.product_id"
    )
)

merge_gold_snapshot(
    source_df=warehouse_inventory_df,
    target_table=warehouse_table,
    target_path=warehouse_path,
    merge_condition=(
        "target.warehouse = source.warehouse"
    )
)

merge_gold_snapshot(
    source_df=low_stock_df,
    target_table=low_stock_table,
    target_path=low_stock_path,
    merge_condition=(
        "target.product_id = source.product_id"
    )
)